In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # fixed seed = reproducible results

N_HOUSEHOLDS = 50
N_MONTHS = 12
STATES = ["Stable", "Strained", "Critical"]

CATEGORIES = [
    "housing", "food_groceries", "transport", "utilities", "airtime_data",
    "mobile_money_transfers", "loan_repayment", "school_fees",
    "entertainment", "savings", "informal_cash_income",
    "transaction_fees", "investment_vehicle",
]

# KIHBS-informed category shares (directional priors, not literal microdata)
BASE_CATEGORY_SHARE = {
    "housing": 0.21, "food_groceries": 0.29, "transport": 0.10, "utilities": 0.06,
    "airtime_data": 0.04, "mobile_money_transfers": 0.03, "loan_repayment": 0.08,
    "school_fees": 0.10, "entertainment": 0.04, "savings": 0.02, "investment_vehicle": 0.02,
}

FEE_BEARING_CATEGORIES = ["mobile_money_transfers", "loan_repayment", "utilities", "housing"]
FEE_RATE = 0.015
FEE_MIN, FEE_MAX = 5.0, 50.0

TIER_PARAMS = {
    "formal_salaried": {
        "income_mean": 65000, "income_sd": 15000, "income_formality": "formal",
        "state_prior": [0.7, 0.25, 0.05],
        "transition": [[0.85, 0.13, 0.02], [0.20, 0.65, 0.15], [0.05, 0.30, 0.65]],
    },
    "informal_irregular": {
        "income_mean": 22000, "income_sd": 9000, "income_formality": "informal",
        "state_prior": [0.30, 0.45, 0.25],
        "transition": [[0.60, 0.30, 0.10], [0.25, 0.50, 0.25], [0.10, 0.35, 0.55]],
    },
}

STATE_EXPENDITURE_RATIO = {"Stable": (0.65, 0.85), "Strained": (0.90, 1.05), "Critical": (1.05, 1.35)}

In [2]:
import sys
print(sys.executable)

/Users/kevinkaniaru/ISPROJTEST/.venv/bin/python


In [ ]:
import math

def one_off_purchase(hid, price, months=6, monthly_interest=0.0):
    """A phone, fridge, laptop: score paying in instalments, and how long saving up would take."""
    if monthly_interest > 0:
        r = monthly_interest
        per_month = price * r / (1 - (1 + r) ** -months)   # standard loan instalment formula
    else:
        per_month = price / months
    left = affordability(hid)["before"]["left_over_KES"]
    return {"price_KES": round(price),
            "instalment_per_month_KES": round(per_month),
            "if_paid_in_instalments": explain(hid, {"loan_repayment": per_month}),
            "months_to_save_up_first": math.ceil(price / left) if left > 0 else "not possible from current left-over"}

def savings_goal(hid, extra_per_month):
    """Saving more moves left-over money into savings; it is not a new cost, so it is not run through the score."""
    left = affordability(hid)["before"]["left_over_KES"]
    if extra_per_month <= left:
        return {"verdict": "Affordable",
                "reasons": [f"You have KES {left:,} left each month, so saving KES {extra_per_month:,} more fits."]}
    short = round(extra_per_month - left)
    return {"verdict": "Not affordable yet",
            "reasons": [f"You have KES {left:,} left each month, so you are KES {short:,} short.",
                        "You would need to cut other spending to reach this goal."]}

print("phone, KES 30,000 over 6 months:")
print(one_off_purchase(hid, 30000, months=6))
print("\nsave KES 3,000 more a month:", savings_goal(hid, 3000))
print("save KES 6,000 more a month:", savings_goal(hid, 6000))